# Chapter 16 — 정오탐 분석과 탐지 개선 (실습)



In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib
# !pip install -q scipy
# !pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
rng = np.random.default_rng(42)

# 데이터 — ch09~11 형식
def gen(n, label, dur, pkts, syn):
    return pd.DataFrame({
        'log_dur':  np.log1p(rng.uniform(*dur, size=n)),
        'log_pkts': np.log1p(rng.integers(*pkts, size=n)),
        'syn':      rng.uniform(*syn, size=n),
        'label':    label,
    })
df = pd.concat([
    gen(16000,'BENIGN',(1000,1_000_000),(5,100),(0,0.05)),
    gen( 2300,'ATTACK',(10,5000),(500,5000),(0.4,0.7)),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)
df['y'] = (df['label']=='ATTACK').astype(int)

# --- 현실화: 정상/공격 경계 중첩 (정오탐이 실제로 존재해야 이 챕터가 의미가 있다) ---
_atk = df.index[df['y']==1].to_numpy()
_b1 = rng.choice(_atk, size=int(len(_atk)*0.25), replace=False)   # 공격 25%: 정상처럼(낮은 syn·pkts)
df.loc[_b1,'syn']      = rng.uniform(0, 0.1, size=len(_b1))
df.loc[_b1,'log_pkts'] = np.log1p(rng.integers(5, 150, size=len(_b1)))
_nrm = df.index[df['y']==0].to_numpy()
_b2 = rng.choice(_nrm, size=int(len(_nrm)*0.10), replace=False)   # 정상 10%: 공격처럼(높은 syn)
df.loc[_b2,'syn']      = rng.uniform(0.35, 0.7, size=len(_b2))
_flip = rng.choice(df.index.to_numpy(), size=int(len(df)*0.04), replace=False)  # 라벨 노이즈 4%
df.loc[_flip,'y'] = 1 - df.loc[_flip,'y']

X_tr, X_te, y_tr, y_te = train_test_split(df[['log_dur','log_pkts','syn']], df['y'],
                                            test_size=0.3, stratify=df['y'], random_state=42)
X_tr, X_valid, y_tr, y_valid = train_test_split(X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=43)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
valid_proba = model.predict_proba(X_valid)[:, 1]

---
## Step 1. 오탐·미탐 심층 분석

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 오탐·미탐 심층 분석
# 만들 결과: pred, y_arr, fp_mask, fn_mask, fp_df, fn_df
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
pred = ...
y_arr = ...
fp_mask = ...
fn_mask = ...
fp_df = ...
fn_df = ...
print(f'FP = ...
...
print(fp_df.sort_values('prob', ascending = ...
...
print(fn_df.sort_values('prob', ascending = ...
```
</details>


---
## Step 2. Threshold 튜닝 — 50개

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. Threshold 튜닝 — 50개
# 만들 결과: thresholds, rows, pred_t, tune
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
thresholds = ...
rows = ...
for t in thresholds:
    pred_t = ...
    ...
        'precision': precision_score(y_valid.values, pred_t, zero_division = ...
        'recall':    recall_score(y_valid.values, pred_t, zero_division = ...
        'f1':        f1_score(y_valid.values, pred_t, zero_division = ...
        ...
tune = ...

fig, ax = ...
ax.plot(tune['threshold'], tune['precision'], label = ...
ax.plot(tune['threshold'], tune['recall'],    label = ...
ax.plot(tune['threshold'], tune['f1'],        label = ...
...
```
</details>


---
## Step 3. 시나리오별 최적 임계값

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 시나리오별 최적 임계값
# 만들 결과: scenarios, b
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
scenarios = ...
    ...
    {'scenario':'best f1 (균형)',              'best': tune.sort_values('f1', ascending = ...
...
for _, r in scenarios.iterrows():
    b = ...
    if b is not None:
        print(f"{r['scenario']}: t = ...
```
</details>


In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 시나리오별 최적 임계값
# 만들 결과: tune['cost'], best, final_pred
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 비용 기반 최적 임계값 — FP 와 FN 의 실제 비용이 다를 때 '총비용 최소'를 고른다.
C_FP, C_FN = ...
tune['cost'] = ...
best = ...
print(f"비용비 FP:FN = ...
      f"(FP = ...
fig, ax = ...
ax.plot(tune['threshold'], tune['cost'], color = ...
ax.axvline(best['threshold'], ls = ...
ax.set_title('임계값별 총비용 (C_FP = ...
...
# 고정한 임계값으로 보류 테스트셋을 평가한다.
final_pred = ...
...
```
</details>


---
## Step 4. 드리프트 모니터링 — KS 검정

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 드리프트 모니터링 — KS 검정
# 만들 결과: X_prod, flag, proba_tr, proba_pr
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 운영 데이터: 분포가 약간 바뀐 신규 데이터
X_prod = ...
    'log_dur':  np.log1p(rng.uniform(500, 800_000, size = ...
    'log_pkts': np.log1p(rng.integers(10, 200, size = ...
    'syn':      rng.uniform(0.05, 0.15, size = ...
...
for col in X_tr.columns:
    stat, p = ...
    flag = ...
    print(f'{col:10s}  KS = ...

# 출력 분포 드리프트
proba_tr = ...
proba_pr = ...
print(f'\n출력 평균 — train = ...
```
</details>


---
## Step 5. 재학습 시뮬레이션

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. 재학습 시뮬레이션
# 만들 결과: X_prod['y'], new_train, model2, auc_old, auc_new
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 새 라벨 데이터 추가 (운영 분석가 피드백 시뮬레이션)
X_prod['y'] = ...
new_train = ...
X_new, y_new = ...
model2 = ...

# A/B 평가
auc_old = ...
auc_new = ...
from sklearn.metrics import f1_score
print(f'F1 old = ...
print(f'F1 new = ...
```
</details>


---
## 적대적 머신러닝 — 회피 공격 데모
공격 샘플을 조금씩 변형해 탐지기가 '정상'으로 오분류하게 만든다(evasion). 방어: 적대적 훈련·입력 정규화·앙상블·모니터링. 표준 참조 MITRE ATLAS / OWASP ML Top 10.


In [ ]:
# ── 실습 과제 ──
# 목표: 적대적 머신러닝 — 회피 공격 데모 — 공격 샘플을 조금씩 변형해 탐지기가 '정상'으로 오분류하게 만든다(evasion). 방어: 적대적 훈련·입력 정규화·앙상블·모니터링. 표준 참조 MITRE ATLAS / OWASP ML Top 10.
# 만들 결과: cols, normal_mean, cand, orig, x, flipped
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
import pandas as pd
cols = ...
normal_mean = ...
cand = ...
orig = ...
while model.predict(pd.DataFrame([x], columns=cols))[0] == 1 and alpha < 1.0:
    alpha += ...
    x = ...
flipped = ...
...
```
</details>


---
## 정리
- 임계값 50개 변화로 운영 시나리오에 맞추기가 즉시 가능.
- KS 검정으로 드리프트 자동 감지.
- 재학습 + A/B 비교가 운영 ML 의 표준 사이클.
- 다음 챕터 (ch17) 부터 LLM 으로 같은 로그를 자연어로 해석한다.